In [0]:
# Criação dos schemas por camada (roda uma vez só, idempotente)
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
from pyspark.sql import functions as F

df = spark.read.option("recursiveFileLookup", "true").option("multiLine", True) \
    .json('/Volumes/workspace/default/mtgo/PUC-RIO_MVP_MTGO/')

df_bronze = df.withColumn("arquivo_origem", F.col("_metadata.file_path")) \
              .withColumn("data_ingestao", F.current_timestamp()) \
              .withColumn("fonte", F.lit("mtgo.com"))

df_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.mtgo_torneios_raw")

In [0]:
spark.table("workspace.bronze.mtgo_torneios_raw").count()

In [0]:
spark.table("workspace.bronze.mtgo_torneios_raw").printSchema()

In [0]:
display(spark.table("workspace.bronze.mtgo_torneios_raw"))

## Ingestão do Scryfall (fonte de enriquecimento de DIM_CARTA)
Download do bulk data "Oracle Cards" do Scryfall e persistência como
Bronze, sem nenhum tratamento — mesma lógica aplicada aos dados do mtgo.com.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.scryfall;
CREATE VOLUME IF NOT EXISTS workspace.scryfall.dados;

In [0]:
import requests

headers = {
    "User-Agent": "PUC-RIO-MVP-EngDados/1.0",
    "Accept": "application/json"
}

resp = requests.get("https://api.scryfall.com/bulk-data/oracle-cards", headers=headers)
print(resp.status_code)
print(resp.json())

In [0]:
download_url = resp.json()["jsonl_download_uri"]

dbutils.fs.cp(download_url, "/Volumes/workspace/scryfall/dados/oracle-cards.jsonl.gz")

In [0]:
from pyspark.sql import functions as F

df_scryfall_raw = spark.read.json("/Volumes/workspace/scryfall/dados/oracle-cards.jsonl.gz")

df_scryfall_bronze = df_scryfall_raw.withColumn("arquivo_origem", F.col("_metadata.file_path")) \
                                     .withColumn("data_ingestao", F.current_timestamp()) \
                                     .withColumn("fonte", F.lit("scryfall.com"))

df_scryfall_bronze.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.scryfall_cartas_raw")

df_scryfall_bronze.count()